In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(os.path.abspath(".."))
import src.raw_preprocessing as rp
import src.feature_engineering as fe
import src.create_dataset as cd
from datetime import datetime, timedelta

In [59]:
conso = pd.read_parquet("../data/final_datasets/datasets_linear_models/conso_v3_linear.parquet")

In [60]:
conso.columns

Index(['Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '59T', '75T', '13T', '33T', 'T', 'U', 'FF', 'PMER', 'RR1',
       'year', 'is_weekend', 'hour_sin', 'hour_cos', 'day_of_week_sin',
       'day_of_week_cos', 'month_sin', 'month_cos', 'lagged_1', 'lagged_2',
       'lagged_48', 'lagged_336', 'rolling_mean_24h', 'rolling_std_24h',
       'rolling_mean_7d', 'rolling_std_7d', 'rolling_max_24h',
       'rolling_min_24h', 'consumption_diff_1', 'consumption_diff_48',
       'consumption_pct_change_1', 'consumption_pct_change_48',
       'season_Spring', 'season_Summer', 'season_Winter', 'temp_sq',
       'humidity_sq', 'hour_x_is_weekend', 'hour_x_is_holiday', 'hour_x_dow',
       'hour_x_month', 'is_weekend_x_month', 'is_holiday_x_month',
       'hour_x_temp', 'hour_x_humidity', 'hour_x_wind', 'is_weekend_x_temp',
       'is_ho

In [ ]:
cd.download_monthly_data()

In [61]:
df = rp.conso_preprocess(Path("../data/conso/real_time_conso/"))

df = df[df["Heures"].apply(lambda x: x.minute in {00, 30})]

df = df.reset_index(drop=True)
df.loc[len(df)] = None

In [62]:
df

,Date,Heures,Consommation
0,2026-05-01,00:00:00,41688.0
1,2026-05-01,00:30:00,40393.0
2,2026-05-01,01:00:00,38108.0
3,2026-05-01,01:30:00,37744.0
4,2026-05-01,02:00:00,36708.0
...,...,...,...
3818,2026-07-19,13:00:00,42182.0
3819,2026-07-19,13:30:00,41187.0
3820,2026-07-19,14:00:00,41283.0
3821,2026-07-19,14:30:00,41408.0


In [63]:
df = fe.lagged_consumption(df)

In [64]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN
...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0


In [65]:
t = df["Heures"].iloc[-2]

new_time = (
    datetime.combine(datetime.today(), t)
    + timedelta(minutes=30)
).time()

In [66]:
new_time

datetime.time(15, 0)

In [67]:
df.loc[len(df)-1, "Heures"] = new_time

In [68]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN
...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0


In [70]:
from datetime import date
from vacances_scolaires_france import SchoolHolidayDates

vacances = SchoolHolidayDates()
today = date.today()

zone_a = vacances.is_holiday_for_zone(today, "A")
zone_b = vacances.is_holiday_for_zone(today, "B")
zone_c = vacances.is_holiday_for_zone(today, "C")

print(f"Zone A : {zone_a}")
print(f"Zone B : {zone_b}")
print(f"Zone C : {zone_c}")

Zone A : True
Zone B : True
Zone C : True


### RTE France API

In [86]:
id_client = "863fa354-33b4-4bf6-a844-3dd668062f92"
id_secret = "83b2284d-b40f-4627-916f-d35473030cbf"
url = "https://digital.iservices.rte-france.com/token/oauth/"

response = requests.post(url, auth=(id_client, id_secret))

In [87]:
response.json()

{'access_token': 'k474YPtYGg8Im6fzkvcoIUeWaPI4fzxRjj6ankSZK5EghTijE48cHG',
 'token_type': 'Bearer',
 'expires_in': 3600}

In [88]:
token = response.json()["access_token"]
headers = {
    "Authorization" : f"Bearer {token}"
}

url = "https://digital.iservices.rte-france.com/open_api/consumption/v1/short_term"

data = requests.get(url, headers=headers)

In [89]:
data.json()

{'short_term': [{'type': 'REALISED',
   'start_date': '2026-07-17T00:00:00+02:00',
   'end_date': '2026-07-18T00:00:00+02:00',
   'values': [{'start_date': '2026-07-17T00:00:00+02:00',
     'end_date': '2026-07-17T00:15:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 47387},
    {'start_date': '2026-07-17T00:15:00+02:00',
     'end_date': '2026-07-17T00:30:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 46988},
    {'start_date': '2026-07-17T00:30:00+02:00',
     'end_date': '2026-07-17T00:45:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 45765},
    {'start_date': '2026-07-17T00:45:00+02:00',
     'end_date': '2026-07-17T01:00:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 44762},
    {'start_date': '2026-07-17T01:00:00+02:00',
     'end_date': '2026-07-17T01:15:00+02:00',
     'updated_date': '2026-07-17T13:05:49+02:00',
     'value': 43709},
    {'start_date': '2026-07-17T01

### Open-Meteo API

In [61]:
token_climat = "0qQ6Dilvnsvc1wktvJ5EZdeef1WNy1JVPNGLqWlzBchwzHjP3iMdg"

headers_clim = {
    "Authorization" : f"Bearer {token_climat}"
}

url_temp = "http://www.infoclimat.fr/public-api/gfs/json?_ll=48.85341,2.3488&_auth=U0kCFQ9xASNTflBnAHYDKgJqDjsJfwIlA39QM1g9USxROgNiUjIEYlM9BnsCLQI0VnsPbA41VWUDaFIqCXsHZlM5Am4PZAFmUzxQNQAvAygCMA5uCTUCOANgUDRYKlEtUTMDZ1I0BHhTPQZtAjMCKFZsD2YOL1VoA2JSKgl7B2VTNwJuD2gBYVM8UDMANAM3AjkOcQkpAjwDMVAyWD1RYVEyAzRSMQQyUzQGMgJnAjdWbQ9xDjlVbwNhUj0JZgdlUzACYw9zAXxTRVBBAC0DdwJzDjsJcAInAzVQaVhh&_c=375f0112f9eb404558d62deef26d2dc5"

data_clim = requests.get(url_temp, headers=headers_clim)

In [62]:
data_clim.json()

{'request_state': 200,
 'request_key': 'fd543c77e33d6c8a5e218e948a19e487',
 'message': 'OK',
 'model_run': '02',
 'source': 'internal:GFS:1',
 '2026-07-16 05:00:00': {'temperature': {'2m': 292.3,
   'sol': 292.8,
   '500hPa': 273.2,
   '850hPa': 273.2},
  'pression': {'niveau_de_la_mer': 101710},
  'pluie': 0,
  'pluie_convective': 0,
  'humidite': {'2m': 48.8},
  'vent_moyen': {'10m': 8.2},
  'vent_rafales': {'10m': 11.9},
  'vent_direction': {'10m': 375},
  'iso_zero': 3623,
  'risque_neige': 'non',
  'cape': 0,
  'nebulosite': {'haute': 0, 'moyenne': 0, 'basse': 0, 'totale': 18}},
 '2026-07-16 08:00:00': {'temperature': {'2m': 294.3,
   'sol': 293.3,
   '500hPa': 273.2,
   '850hPa': 273.2},
  'pression': {'niveau_de_la_mer': 101730},
  'pluie': 0,
  'pluie_convective': 0,
  'humidite': {'2m': 47.9},
  'vent_moyen': {'10m': 6},
  'vent_rafales': {'10m': 8.4},
  'vent_direction': {'10m': 383},
  'iso_zero': 3569,
  'risque_neige': 'non',
  'cape': 0,
  'nebulosite': {'haute': 0, 'moye

In [67]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 45.7491,
	"longitude": 4.8479,
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure", "wind_speed_10m"],
	"timezone": "Europe/London",
	"past_days": 7,
	"forecast_days": 1,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 45.7400016784668°N 4.839999675750732°E
Elevation: 176.0 m asl
Timezone: b'Europe/London'b'GMT+1'
Timezone difference to GMT+0: 3600s

Hourly data
                          date  temperature_2m  relative_humidity_2m  rain  \
0   2026-07-09 00:00:00+01:00       27.513000                  40.0   0.0   
1   2026-07-09 01:00:00+01:00       26.863001                  43.0   0.0   
2   2026-07-09 02:00:00+01:00       26.163000                  46.0   0.0   
3   2026-07-09 03:00:00+01:00       25.413000                  49.0   0.0   
4   2026-07-09 04:00:00+01:00       24.863001                  49.0   0.0   
..                        ...             ...                   ...   ...   
187 2026-07-16 19:00:00+01:00       23.263000                  67.0   2.1   
188 2026-07-16 20:00:00+01:00       22.913000                  72.0   0.1   
189 2026-07-16 21:00:00+01:00       22.613001                  72.0   0.0   
190 2026-07-16 22:00:00+01:00       22.463001                  74.0   

In [71]:
hourly_dataframe

,date,temperature_2m,relative_humidity_2m,rain,surface_pressure,wind_speed_10m
0,2026-07-09 00:00:00+01:00,27.513000,40.0,0.0,995.333679,10.315115
1,2026-07-09 01:00:00+01:00,26.863001,43.0,0.0,994.996765,10.948973
2,2026-07-09 02:00:00+01:00,26.163000,46.0,0.0,995.342224,11.792404
3,2026-07-09 03:00:00+01:00,25.413000,49.0,0.0,994.998108,11.384198
4,2026-07-09 04:00:00+01:00,24.863001,49.0,0.0,994.667480,11.019764
...,...,...,...,...,...,...
187,2026-07-16 19:00:00+01:00,23.263000,67.0,2.1,996.323425,8.654986
188,2026-07-16 20:00:00+01:00,22.913000,72.0,0.1,995.515747,2.305125
189,2026-07-16 21:00:00+01:00,22.613001,72.0,0.0,996.082947,5.495161
190,2026-07-16 22:00:00+01:00,22.463001,74.0,0.0,997.640808,10.105681
